# 시계열 검증과 정보 누수

## Bike Sharing 수요의 과거–미래 검증

정확한 1시간 lag를 만들고 무작위 분할과 시간순 분할의 성능 차이를 persistence baseline과 함께 비교합니다.


## 이번 실습에서 완성할 것

- 정확히 1시간 전 timestamp를 연결합니다.
- 무작위·시간순 분할의 실제 기간을 비교합니다.
- 직전 값 기준모형과 선형모형을 같은 미래행에서 비교합니다.

**완료 결과:** 누락시각 lag 표, 분할 기간표, 미래평가 MAE와 정보 가용성 문장


## 이번 교시에서 사용할 데이터

### 강의 시연: Steel 15분 시계열

UCI Steel 자료의 35,040개 관측을 `date` 순서로 정렬합니다. 한 행은 15분 구간이며, 직전 15분 사용량을 정확한 timestamp join으로 연결해 one-step-ahead feature를 만듭니다.

- **예측 대상:** 다음 15분의 `Usage_kWh`
- **시간 feature:** 예측 시점 이전에 알려진 lag와 달력정보
- **비교할 split:** random split과 과거 학습–미래 평가 split
- **baseline:** 직전 관측을 그대로 사용하는 persistence

### 적용 실습: Bike 시간별 시계열

17,379개의 시간별 대여 관측을 날짜와 시간으로 결합해 timestamp를 만듭니다. 정확히 한 시간 전 `cnt`가 존재할 때만 lag를 연결하고, 앞 기간으로 학습해 뒤 기간을 평가합니다. 관측이 빠진 시간에는 단순 행 이동이 정확한 1시간 lag가 아닐 수 있으므로 timestamp 차이를 확인합니다.


## 실습에서 고정할 연구조건

| 그대로 유지할 방법 | Bike에서 새로 확인할 조건 |
|---|---|
| timestamp로 정확한 과거값 연결 | 예측 기준시점과 1시간 예측간격 |
| 과거 학습–미래 평가 분할 | 결과시각 날씨정보의 실제 확보시점 |
| 직전 값 기준모형과 MAE | 무작위 분할과 시간순 분할이 답하는 질문의 차이 |

여러 설정을 탐색하지 않고 Steel에서 배운 검증절차를 Bike의 한 시간 단위에 적용합니다.


## 실제 실행 순서와 산출물

1. Bike timestamp와 정확히 1시간 전 `cnt`를 연결합니다.
2. 무작위 분할과 시간순 분할의 실제 날짜 범위를 출력합니다.
3. 같은 미래행에서 persistence와 선형모형을 비교합니다.
4. 입력정보 가용성 표와 조건부 benchmark 한계를 기록합니다.

**산출물:** 누락시각 lag 표, 분할 기간표, 세 결과의 MAE, 입력정보 가용성 기록


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 정확한 과거 시점의 lag 연결

관측 순서를 정렬하고 timestamp join으로 바로 이전 시간의 target만 feature로 만듭니다. 행 shift가 실제 시간 lag와 같은지 가정하지 않습니다.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

raw = pd.read_csv(find_course_data("bike/hour.csv"), parse_dates=["dteday"])
raw["timestamp"] = raw["dteday"] + pd.to_timedelta(raw["hr"], unit="h")
raw = raw.sort_values("timestamp")
target_name = "cnt"
analysis_name = "Bike one-hour time validation"
lag = raw[["timestamp", target_name]].copy()
lag["timestamp"] = lag["timestamp"] + pd.Timedelta(hours=1)
lag = lag.rename(columns={target_name: "lag_1h"})
data = raw.merge(lag, on="timestamp", how="left").dropna(subset=["lag_1h"]).copy()
data["hour_sin"] = np.sin(2 * np.pi * data["hr"] / 24)
data["hour_cos"] = np.cos(2 * np.pi * data["hr"] / 24)
features = ["lag_1h", "hour_sin", "hour_cos", "workingday", "temp", "hum", "windspeed"]
TARGET_TIME_WEATHER_COLUMNS = ['temp', 'hum', 'windspeed']
TARGET_TIME_WEATHER_USED = sorted(set(features) & set(TARGET_TIME_WEATHER_COLUMNS))
WEATHER_BENCHMARK_NOTE = '이 결과는 결과시각의 실제 날씨 관측값을 사용할 수 있다고 가정한 조건부 benchmark입니다. 실제 운영 예측에는 예보값과 예보오차 검증이 필요합니다.'
feature_availability = pd.DataFrame({
    '입력': ['lag_1h', 'workingday', *TARGET_TIME_WEATHER_COLUMNS],
    '값의 기준시각': ['1시간 전', '결과시각', '결과시각', '결과시각', '결과시각'],
    '예측 때 확보': ['직전 측정이 즉시 들어온다고 가정', '달력으로 사전 확인', '실측값만 제공', '실측값만 제공', '실측값만 제공'],
    '판정': ['가정 아래 사용', '사용 가능', '예보자료 필요', '예보자료 필요', '예보자료 필요'],
})
print('\n[입력정보의 실제 확보시점]')
display(feature_availability)
print(WEATHER_BENCHMARK_NOTE)

print("rows after exact lag join:", len(data))
print("time range:", data["timestamp"].min(), data["timestamp"].max())
display(data[["timestamp", target_name, *features]].head(3))
missing_time_example = pd.DataFrame({
    "timestamp": pd.to_datetime(["2026-01-01 08:00", "2026-01-01 10:00"]),
    "수요": [17.0, 31.0],
})
missing_time_example["앞 행의 수요"] = missing_time_example["수요"].shift(1)
exact_source = missing_time_example[["timestamp", "수요"]].copy()
exact_source["timestamp"] = exact_source["timestamp"] + pd.Timedelta(hours=1)
exact_source = exact_source.rename(columns={"수요": "정확히 1시간 전 수요"})
missing_time_example = missing_time_example.merge(exact_source, on="timestamp", how="left")
print("\n[09:00이 빠진 경우: 앞 행과 정확히 1시간 전은 다릅니다]")
display(missing_time_example)

print("\n[현재 시각과 정확히 1시간 전 값]")
lag_view = data[["timestamp", target_name, "lag_1h"]].head(8).copy()
lag_view["현재-1시간전"] = lag_view[target_name] - lag_view["lag_1h"]
display(lag_view)
print("단순히 앞 행을 복사한 것이 아니라 timestamp를 1시간 이동해 연결했습니다.")


## 2. 무작위 분할과 시간순 분할

동일 모델을 무작위 분할과 과거→미래 분할에서 비교합니다. chronological cut 이후 test timestamp가 모두 train보다 늦어야 합니다.


In [ ]:
X, y = data[features], data[target_name]
cut = int(len(data) * 0.80)
X_train_t, X_test_t = X.iloc[:cut], X.iloc[cut:]
y_train_t, y_test_t = y.iloc[:cut], y.iloc[cut:]
time_train, time_test = data["timestamp"].iloc[:cut], data["timestamp"].iloc[cut:]

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.20, random_state=42)
print("chronological train end:", time_train.max())
print("chronological test start:", time_test.min())
random_train_time = data.loc[X_train_r.index, "timestamp"]
random_test_time = data.loc[X_test_r.index, "timestamp"]
split_time_view = pd.DataFrame({
    "분할": ["시간순 train", "시간순 test", "무작위 train", "무작위 test"],
    "첫 시각": [time_train.min(), time_test.min(), random_train_time.min(), random_test_time.min()],
    "마지막 시각": [time_train.max(), time_test.max(), random_train_time.max(), random_test_time.max()],
    "행 수": [len(X_train_t), len(X_test_t), len(X_train_r), len(X_test_r)],
})
print("\n[분할 뒤 시간 범위]")
display(split_time_view)


## 3. Persistence와 선형모형 비교

chronological test에서 직전 관측을 그대로 예측하는 persistence와 선형모형을 비교하고, random split 점수와 나란히 둡니다.


In [ ]:
model_time = LinearRegression().fit(X_train_t, y_train_t)
model_random = LinearRegression().fit(X_train_r, y_train_r)
pred_time = model_time.predict(X_test_t)
pred_random = model_random.predict(X_test_r)
persistence = X_test_t["lag_1h"].to_numpy()

result_table = pd.DataFrame([
    {"validation": "chronological", "model": "persistence", "MAE": mean_absolute_error(y_test_t, persistence)},
    {"validation": "chronological", "model": "linear", "MAE": mean_absolute_error(y_test_t, pred_time)},
    {"validation": "random", "model": "linear", "MAE": mean_absolute_error(y_test_r, pred_random)},
])
display(result_table)
prediction_view = pd.DataFrame({
    "timestamp": time_test.iloc[:8].to_numpy(),
    "실제 수요": y_test_t.iloc[:8].to_numpy(),
    "직전 값 baseline": persistence[:8],
    "선형모형 예측": pred_time[:8],
})
prediction_view["baseline 절대오차"] = (
    prediction_view["실제 수요"] - prediction_view["직전 값 baseline"]
).abs()
prediction_view["선형모형 절대오차"] = (
    prediction_view["실제 수요"] - prediction_view["선형모형 예측"]
).abs()
print("\n[같은 미래 시각의 실제값·예측값·절대오차]")
display(prediction_view.round(2))
print("[같은 미래 test 전체의 MAE]")
display(result_table.round(3))
print("\n[해석 조건]", WEATHER_BENCHMARK_NOTE)


## 4. 시계열 평가 조건

과거와 미래가 겹치지 않는지, lag가 target과 같은 행 정보를 복사하지 않았는지, 모든 예측 길이가 맞는지 검사합니다.


In [ ]:
checks = pd.Series({
    "weather benchmark labeled": (not TARGET_TIME_WEATHER_USED) or ("조건부 benchmark" in WEATHER_BENCHMARK_NOTE),
    "strict time order": time_train.max() < time_test.min(),
    "no missing features": not X.isna().any().any(),
    "prediction length": len(pred_time) == len(y_test_t),
    "three comparison rows": len(result_table) == 3,
    "finite MAE": np.isfinite(result_table["MAE"]).all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("시계열 순서 검증: PASS")


## 연구 기록

다음 다섯 항목을 Notebook의 실제 출력으로 작성합니다.

1. **예측 기준시점과 예측간격:**
2. **예측 때 이용할 수 있다고 판단한 입력과 근거:**
3. **시간순 평가의 persistence MAE와 선형모형 MAE:**
4. **무작위 분할 점수를 미래예측 성능으로 보고하지 않는 이유:**
5. **아직 검증하지 않은 시기·계절 또는 정보지연:**

### 선택 검토: ChatGPT에 물어볼 때

학습·평가기간, 예측간격, 입력별 기준시각·확보시각 표와 MAE를 붙여 넣습니다.

```text
아래에 제공한 정보만 사용해 입력별로 ‘사용 가능 / 사용 불가 / 정보 부족’을 판정해 주세요. 가장 의심되는 시간정보 한 가지와 Notebook에서 직접 확인할 출력 한 가지를 제시해 주세요. 제공되지 않은 날씨예보나 측정지연은 추정하지 마세요.
```


## 완료 확인

마지막 출력이 `시계열 순서 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- 예측 기준시점과 예측간격은 무엇이며, 각 lag는 정확히 어느 timestamp의 값입니까?
- 학습 종료시각과 평가 시작시각은 시간 순서를 지킵니까?
- 직전 값 기준모형과 후보를 같은 미래행에서 비교했고, 실제 운영 전에 확인할 정보 가용성은 무엇입니까?
